In [1]:

import json
import pennylane as qml
import pennylane.numpy as np

def U():
    """U function that creates the gate that checks the parity of the number of forests.
    It should not return anything, you simply need to add the gates."""

    # Auxiliary qubit for parity is wire 8
    aux_wire = 8

    # Condition 1: If the first qubit (wire 0) is |1>, it marks the start of a forest.
    # Flip the auxiliary qubit (wire 8) if wire 0 is |1>.
    qml.CNOT(wires=[0, aux_wire])

    # Condition 2: For each subsequent qubit, if the current qubit (wire i) is |1>
    # AND the previous qubit (wire i-1) is |0>, it marks the start of a new forest.
    # Iterate from wire 1 up to wire 7.
    for i in range(1, 8):
        # We need to detect the pattern |01> on wires [i-1, i].
        # To control on |0> state of wire (i-1), apply an X gate to wire (i-1).
        # This transforms |0> to |1> and |1> to |0>.
        qml.PauliX(wires=i-1)
        
        # Now, if the original state of wire (i-1) was |0> (now |1>) AND
        # wire i is |1>, then apply a CNOT to the auxiliary qubit.
        # This is achieved with a Toffoli gate (CCNOT).
        qml.Toffoli(wires=[i-1, i, aux_wire])
        
        # Revert wire (i-1) to its original state by applying another X gate.
        # This ensures the input qubits are not permanently altered.
        qml.PauliX(wires=i-1)


# These functions are responsible for testing the solution.


def run(test_case_input: str) -> str:

    input = json.loads(test_case_input)
    wires_input = [0,1,2,3,4,5,6,7]

    # The circuit has 8 input qubits (0-7) and 1 auxiliary qubit (8), for a total of 9 wires.
    # Setting shots to 100 for more stable probability measurements, though for default.qubit
    # without actual hardware, exact probabilities are always returned.
    dev = qml.device("default.qubit", wires = 9, shots = 100)

    @qml.qnode(dev)
    def circuit():
      # Embed the input list into the quantum state on wires 0-7.
      qml.BasisEmbedding(input, wires = wires_input)

      # Apply the U operation to determine the parity of forests.
      U()

      # Return the probabilities of the auxiliary qubit (wire 8).
      # The probability of |1> on wire 8 will be 1 if the parity is odd, and 0 if even.
      return qml.probs(wires = 8)

    # Get the probability of measuring |1> on the auxiliary qubit.
    # If this is close to 1, the parity is odd (1). If close to 0, the parity is even (0).
    return str(float(circuit()[1]))


def check(have: str, want: str) -> None:
    # Convert the expected output (string '0' or '1') to a float probability.
    want_prob = float(want)

    # Compare the actual probability with the expected probability with a tolerance.
    assert np.isclose(float(have), want_prob, atol=1e-6), f"Wrong answer! Expected probability of 1 is {want_prob}, but got {float(have)}"

